# 01 - Get the research datasets

Acquire the datasets in the research presentation, keeping original releases under `datasets/`.

| Dataset | Research role | Official access |
| --- | --- | --- |
| [PIDray](https://github.com/lutao2021/PIDray) | Main training set; Easy, Hard, Hidden test splits | Google Drive / Baidu archives |
| [STCray](https://github.com/Divs1159/STING-BEE) | External test only | Gated Hugging Face RAR archives |
| [SIXray](https://github.com/MeioJane/SIXray) | Optional external test | Baidu; published direct mirror currently unavailable |
| [SIXray-D](https://www.ntu.edu.sg/rose/research-focus/datasets) | Optional boxed SIXray subset | Request annotations from the dataset owners |

**Import versus download:** the verified official releases are archives, not a ready-to-stream Hugging Face `datasets` table. We use `huggingface_hub` and `gdown` to acquire the original files, then work with local paths. Importing those libraries does not import the images.

Select the repository's `.venv` Python kernel. Run the setup cells first, then the acquisition cells you need. Large and optional downloads are disabled by default. Tokens must never be pasted into this notebook or saved in its output.

Keep STCray and SIXray/SIXray-D **test-only**: do not train on them or use them to choose the detection threshold. Preserve all official splits. Do not infer a SIXray detection split from its image-level labels.

In [ ]:
import json
import shutil
import warnings
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse
from urllib.request import urlopen

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / '01_get_datasets.ipynb').is_file() and (path / 'datasets').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the Computer_Vision repository.')

DATASETS_DIR = PROJECT_ROOT / 'datasets'
DATASET_DIRS = {name: DATASETS_DIR / name for name in ('pidray', 'stcray', 'sixray', 'sixray_d')}
for directory in DATASET_DIRS.values():
    (directory / 'raw').mkdir(parents=True, exist_ok=True)
    (directory / 'extracted').mkdir(exist_ok=True)

print(f'Dataset storage: {DATASETS_DIR}')
print(f'Free space: {shutil.disk_usage(DATASETS_DIR).free / 1024**3:.1f} GiB')

## Dependencies

The next cell installs into the selected notebook kernel. It does not install the authors' training frameworks, PyTorch, or model weights. Restart the kernel if Jupyter requests it, then rerun the paths cell.

In [ ]:
%pip install -r "{PROJECT_ROOT / 'requirements.txt'}"

In [ ]:
DOWNLOAD_PIDRAY = True
DOWNLOAD_STCRAY = False  # Enable after obtaining Hugging Face access.
DOWNLOAD_SIXRAY = False  # Optional; requires working, approved file URLs below.
INCLUDE_STCRAY_TRAIN = False  # Not needed for our external evaluation.

PIDRAY_DRIVE = 'https://drive.google.com/drive/folders/1zvMIc1bqteRN9Z36hHYpoTGoZArsh4mE'
STCRAY_REPO = 'Naoufel555/STCray-Dataset'

# Populate only with working dataset file URLs obtained from the owners.
# Keys are local filenames; values are HTTPS URLs, not Baidu folder pages.
SIXRAY_FILES = {}

SOURCES = {
    'pidray': 'https://github.com/lutao2021/PIDray',
    'stcray': f'https://huggingface.co/datasets/{STCRAY_REPO}',
    'sixray': 'https://github.com/MeioJane/SIXray',
    'sixray_d': 'https://www.ntu.edu.sg/rose/research-focus/datasets',
}

## PIDray - primary dataset

The presentation and official README report 76,913 train, 24,758 Easy, 9,746 Hard, and 13,069 Hidden scans (124,486 total), with 12 classes and box/mask annotations.

As checked on 2026-11-24, the official Drive folder lists `annotation.tar.gz` and model weights, but no image archive. The next cell downloads dataset archives that are actually listed, **excludes model weights**, and reports exactly what it acquired. This is not a complete dataset until the images are available.

For missing images or Drive quota/access errors, use the authors' [Baidu release](https://pan.baidu.com/s/1qPUNWYItg08pht0kFn8qxg), code `8vz6`, or contact them via the official repository. Place manually acquired archives in `datasets/pidray/raw/`. Do not replace the full release with an unverified small mirror.

In [ ]:
if DOWNLOAD_PIDRAY:
    import gdown

    raw_dir = DATASET_DIRS['pidray'] / 'raw'
    drive_files = gdown.download_folder(
        url=PIDRAY_DRIVE, output=str(raw_dir), skip_download=True, use_cookies=False, timeout=60,
    )
    if not drive_files:
        raise RuntimeError('PIDray Drive listing is empty/unavailable. Use the official Baidu release.')

    archives = [
        file for file in drive_files
        if str(file.path).lower().endswith(('.tar.gz', '.tgz', '.zip', '.rar', '.7z'))
        and 'model' not in str(file.path).lower()
        and 'weight' not in str(file.path).lower()
    ]
    if not archives:
        raise RuntimeError('No PIDray dataset archives found in the official folder.')

    for file in archives:
        destination = raw_dir / Path(file.path).name
        partial = destination.with_name(destination.name + '.part')
        if destination.is_file() and destination.stat().st_size > 0:
            print(f'Already present: {destination.name}')
            continue
        result = gdown.download(id=file.id, output=str(partial), resume=True, use_cookies=False, timeout=60)
        if result is None or not partial.is_file() or partial.stat().st_size == 0:
            raise RuntimeError(f'PIDray download failed: {destination.name}. Check access/quota.')
        partial.replace(destination)
        print(f'Acquired archive: {destination.name}')

    warnings.warn('PIDray is not verified complete. Extract and check images plus annotations below.')
else:
    print('PIDray acquisition disabled; existing local files are unchanged.')

## STCray - external evaluation

1. Open the [official dataset page](https://huggingface.co/datasets/Naoufel555/STCray-Dataset), sign in, accept its terms, and request access if required.
2. In a terminal using `.venv`, run `hf auth login`. Use the hidden token prompt; do not put the token in a cell. Alternatively, supply `HF_TOKEN` through your environment, outside this repository.
3. Set `DOWNLOAD_STCRAY = True` and rerun the configuration and next cell.

The official listing currently has `STCray_TestSet.rar` (~0.98 GB), `STCray_TrainSet.rar` (~1.09 GB), and `STCray_Augmented.rar` (~21.55 GB). We acquire **TestSet only** by default and never automatically acquire the augmented set. The paper reports 16,598 test scans; validate the extracted release rather than assuming its detection annotations or per-image concealment labels are present.

An access error is not a successful download. The Hub library caches/resumes downloads inside this dataset's local directory.

In [ ]:
if DOWNLOAD_STCRAY:
    from huggingface_hub import HfApi, hf_hub_download

    api = HfApi()
    release = api.dataset_info(STCRAY_REPO)
    stcray_revision = release.sha
    requested_files = ['STCray_TestSet.rar', 'ReadMe_STCray.docx']
    if INCLUDE_STCRAY_TRAIN:
        requested_files.append('STCray_TrainSet.rar')
    for filename in requested_files:
        local_file = hf_hub_download(
            repo_id=STCRAY_REPO,
            repo_type='dataset',
            revision=stcray_revision,
            filename=filename,
            local_dir=DATASET_DIRS['stcray'] / 'raw',
        )
        print(f'Acquired: {Path(local_file).name} (revision {stcray_revision})')
    (DATASET_DIRS['stcray'] / 'release.json').write_text(
        json.dumps({'repo_id': STCRAY_REPO, 'revision': stcray_revision, 'files': requested_files}, indent=2),
        encoding='utf-8',
    )
else:
    print('STCray acquisition disabled. Obtain access and enable DOWNLOAD_STCRAY to download.')

## SIXray / SIXray-D - optional

**SIXray:** 1,059,231 images; mostly image-level annotations, with official test bounding boxes. Its license permits **academic use only, no commercial use**. The official [data_list.txt](https://github.com/MeioJane/SIXray/blob/master/data_list.txt) lists annotations, image split files, positive annotations, and 18 multipart image archives (`JPEGImage.tar.gz00` through `17`). The published mirror returned 404 during setup; do not assume those links work.

Use the [official Baidu image release](https://pan.baidu.com/s/1zSbpapRURc9Uzjl-1ZWI_w) and [positive annotations](https://pan.baidu.com/s/1ZNxsYWdVXHyDGG5EvVeadQ), or ask the owners for a working mirror. Store all files in `datasets/sixray/raw/`. For a working direct mirror, populate `SIXRAY_FILES` and enable the cell below. A Baidu share page is **not** a direct file URL. The cell is deliberately disabled until URLs are provided; it will not substitute a different dataset.

**SIXray-D is not interchangeable with SIXray.** Request its detection annotations through [NTU ROSE's dataset catalogue](https://www.ntu.edu.sg/rose/research-focus/datasets). The legacy `rose1.ntu.edu.sg/dataset/SIXray-D.asp` link currently returns 404. Store the approved files and the owners' split instructions in `datasets/sixray_d/`; obtain the associated SIXray images separately. Preserve the supplied train/test definition even though our use is external evaluation only.

In [ ]:
def download_direct(url: str, destination: Path) -> Path:
    if urlparse(url).scheme != 'https':
        raise ValueError('Use a working HTTPS direct file URL from an approved dataset source.')
    if destination.is_file() and destination.stat().st_size > 0:
        print(f'Already present: {destination.name}')
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + '.part')
    with urlopen(url, timeout=60) as response:
        content_type = response.headers.get('Content-Type', '').lower()
        if 'text/html' in content_type:
            raise ValueError(f'The URL returned a web page, not an archive: {destination.name}')
        length = response.headers.get('Content-Length')
        expected_size = int(length) if length is not None else None
        if expected_size is not None and expected_size > shutil.disk_usage(destination.parent).free:
            raise OSError(f'Insufficient disk space for {destination.name}')
        with partial.open('wb') as output:
            shutil.copyfileobj(response, output, length=1024 * 1024)
    actual_size = partial.stat().st_size
    if actual_size == 0 or (expected_size is not None and actual_size != expected_size):
        raise OSError(f'Incomplete download for {destination.name}: {actual_size} bytes')
    partial.replace(destination)
    print(f'Acquired: {destination.name} ({actual_size / 1024**2:.1f} MiB)')
    return destination


if DOWNLOAD_SIXRAY:
    if not SIXRAY_FILES:
        raise ValueError('Provide approved working direct file URLs in SIXRAY_FILES first.')
    for filename, url in SIXRAY_FILES.items():
        if Path(filename).name != filename or filename in ('', '.', '..'):
            raise ValueError(f'Expected a filename without directories: {filename!r}')
        download_direct(url, DATASET_DIRS['sixray'] / 'raw' / filename)
else:
    print('SIXray acquisition disabled; use the official access instructions above.')

## Extract and verify before training

Use your archive application (for example 7-Zip) to extract each release into its matching `datasets/<name>/extracted/` folder, preserving the authors' directory layout. This notebook does not automatically unpack large archives or delete originals. Check free space before extraction. For SIXray, obtain **all 18 parts**, join them in numeric order using your archive tool's combine-files operation, then extract the resulting `JPEGImage.tar.gz`. Do not treat each part as an independent archive.

Read the release's annotation documentation, check that every referenced image exists, confirm box and mask formats, and verify official split counts. In particular, verify PIDray's threat-free split assignment and whether STCray actually releases concealment levels. Do not assume a captions/VQA JSON is a detection-annotation file.

The inventory below distinguishes **archives acquired** from **images extracted**. File counts are only a sanity check, not proof of correct splits or matching annotations. It saves a local manifest in the ignored datasets folder. No dataset is labeled training-ready automatically.

In [ ]:
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
ANNOTATION_SUFFIXES = {'.json', '.xml', '.txt', '.csv'}
manifest = {'checked_at': datetime.now(timezone.utc).isoformat(), 'datasets': {}}

for name, directory in DATASET_DIRS.items():
    raw_files = sorted(path for path in (directory / 'raw').iterdir() if path.is_file())
    acquired = [path for path in raw_files if not path.name.endswith('.part') and path.stat().st_size > 0]
    image_count = 0
    annotation_count = 0
    for path in (directory / 'extracted').rglob('*'):
        if path.is_file():
            image_count += path.suffix.lower() in IMAGE_SUFFIXES
            annotation_count += path.suffix.lower() in ANNOTATION_SUFFIXES
    status = 'needs split/annotation verification' if image_count else 'no extracted images'
    manifest['datasets'][name] = {
        'source': SOURCES[name],
        'raw_files': [{'name': path.name, 'bytes': path.stat().st_size} for path in acquired],
        'partial_files': [path.name for path in raw_files if path.name.endswith('.part')],
        'extracted_images': image_count,
        'annotation_candidates': annotation_count,
        'status': status,
    }
    print(f'{name:9} | acquired files: {len(acquired):2} | images: {image_count:7,} | {status}')
    if not image_count:
        warnings.warn(f'{name}: images are not available in extracted/; this dataset is not ready.')

manifest_path = DATASETS_DIR / 'acquisition_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(f'Inventory saved: {manifest_path}')